# Logistic Regression with PyTorch

In this notebook you will implement a **true binary logistic regression** in PyTorch from scratch, then see how it naturally generalises to the multi-class softmax formulation used in later notebooks.

### Structure
| Part | What you will learn |
|---|---|
| **Part 1** | Binary logistic regression — one output, sigmoid, Binary Cross-Entropy loss |
| **Part 2** | The softmax rewrite — two outputs, categorical CE — and why they are equivalent |

### Important note upfront
The dataset we use (half-moons) has a **non-linear** decision boundary. Logistic regression is a linear classifier — it can only draw a straight line. No matter how long you train it, it will not perfectly fit this data. That failure is intentional: it is the whole motivation for adding hidden layers later.

> **Prerequisite:** Familiarity with basic PyTorch tensors, `nn.Module`, and the concept of backpropagation.

---
# Dependencies and supporting functions

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sklearn.datasets
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

# Plotting helper — do not worry about the internals for now
def plot_decision_boundary(pred_func, X, y, title=''):
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    xx = xx.astype('float32')
    yy = yy.astype('float32')
    Z = pred_func(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure()
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu)
    plt.scatter(X[:, 0], X[:, 1], c=-y, cmap=plt.cm.Spectral)
    if title:
        plt.title(title)

def onehot(t, num_classes):
    out = np.zeros((t.shape[0], num_classes))
    for row, col in enumerate(t):
        out[row, col] = 1
    return out

---
# Dataset: Half-Moons

Two interleaving crescents, each belonging to one class (`0` or `1`). The true boundary between them is curved — already beyond what any linear classifier can represent.

In [ ]:
np.random.seed(0)
num_samples = 300

X, y = sklearn.datasets.make_moons(num_samples, noise=0.20)

X_tr  = X[:100].astype('float32')
X_val = X[100:200].astype('float32')
X_te  = X[200:].astype('float32')

y_tr  = y[:100].astype('int32')
y_val = y[100:200].astype('int32')
y_te  = y[200:].astype('int32')

num_features = X_tr.shape[-1]   # 2

plt.scatter(X_tr[:,0], X_tr[:,1], s=40, c=y_tr, cmap=plt.cm.Spectral)
plt.title('Half-Moon Training Data  (red=class 0, blue=class 1)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

print('X shape:', X.shape, '  y shape:', y.shape)
print('Classes:', np.unique(y))   # 0 and 1 only — binary problem

---
# Part 1 — Binary Logistic Regression

## The model

For a **binary** problem the label $y \in \{0, 1\}$. We only need to predict one number: the **probability of belonging to class 1**.

$$\hat{p} = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}$$

where $\sigma$ is the **sigmoid** function, which squashes any real number into $(0, 1)$.

- $x \in \mathbb{R}^{d}$ — input feature vector
- $w \in \mathbb{R}^{d}$ — weight vector
- $b \in \mathbb{R}$ — scalar bias
- $\hat{p} \in (0, 1)$ — predicted probability of class 1

The **decision rule**: predict class 1 if $\hat{p} \geq 0.5$, else predict class 0.  
This threshold corresponds exactly to the hyperplane $w^\top x + b = 0$ — a straight line in 2D.

In tensor shapes:
- `x` : `[batch_size, num_features]`
- `W` : `[1, num_features]` — **one** output unit
- `b` : `[1]`
- `p̂` : `[batch_size, 1]`

In [ ]:
class BinaryLogisticRegression(nn.Module):
    """
    True binary logistic regression.
    One output unit: the predicted probability P(y=1 | x).
    """

    def __init__(self):
        super(BinaryLogisticRegression, self).__init__()
        # ONE output unit — we only need P(class=1)
        # W shape: [1, num_features]   b shape: [1]
        self.W = nn.Parameter(torch.randn(1, num_features))
        self.b = nn.Parameter(torch.randn(1))

    def forward(self, x):
        # Linear step: z = xW^T + b   shape [batch_size, 1]
        z = F.linear(x, self.W, self.b)
        # Sigmoid squashes z to (0, 1)
        return torch.sigmoid(z)


net = BinaryLogisticRegression()

print('W shape         :', net.W.shape)   # [1, 2]
print('b shape         :', net.b.shape)   # [1]
print('Total parameters:', sum(p.numel() for p in net.parameters()))  # 3: w1, w2, b

### Quick forward pass check

Before training, verify the output shape and that all values are in $(0, 1)$.

In [ ]:
X_sample = torch.randn(5, num_features)
p_hat = net(X_sample)

print('Input shape  :', X_sample.shape)   # [5, 2]
print('Output shape :', p_hat.shape)       # [5, 1] — one probability per sample
print('Output values:', p_hat.data)        # all in (0, 1)
print('Min / Max    :', p_hat.min().item(), '/', p_hat.max().item())

## Parameters and Gradients

Each `nn.Parameter` carries two tensors:
- `.data` — the current values
- `.grad` — the gradient populated during `.backward()` (None before the first backward pass)

In [ ]:
print('W.data :', net.W.data)
print('W.grad :', net.W.grad)   # None — no backward pass yet

# Trigger a backward pass with a simple scalar loss
dummy_loss = net(X_sample).sum()
dummy_loss.backward()

print('\nAfter backward():')
print('W.grad :', net.W.grad)   # now populated
print('b.grad :', net.b.grad)

# Always zero gradients before the next training step to avoid accumulation
net.zero_grad()
print('\nAfter zero_grad():')
print('W.grad :', net.W.grad)   # back to None / zeros

## Loss Function: Binary Cross-Entropy (BCE)

For binary classification the loss is:

$$\mathcal{L}_{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \Big[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \Big]$$

Intuitively:
- When $y_i = 1$: we want $\hat{p}_i \to 1$, so $-\log(\hat{p}_i)$ should be small
- When $y_i = 0$: we want $\hat{p}_i \to 0$, so $-\log(1 - \hat{p}_i)$ should be small

Notice: labels are **scalar** 0/1 — no one-hot encoding needed.

In [ ]:
def binary_cross_entropy(p_hat, y):
    """
    p_hat : [batch_size, 1] — predicted probabilities from sigmoid
    y     : [batch_size, 1] — true binary labels (0.0 or 1.0)
    """
    p_hat = torch.clamp(p_hat, 1e-7, 1 - 1e-7)   # avoid log(0)
    loss  = -(y * torch.log(p_hat) + (1 - y) * torch.log(1 - p_hat))
    return loss.mean()


# Sanity checks
y_true    = torch.tensor([[1.0], [0.0], [1.0]])

p_perfect = torch.tensor([[1.0], [0.0], [1.0]])
p_random  = torch.tensor([[0.5], [0.5], [0.5]])
p_wrong   = torch.tensor([[0.0], [1.0], [0.0]])

print(f'Perfect prediction  loss: {binary_cross_entropy(p_perfect, y_true).item():.4f}  (expect ~0)')
print(f'Chance-level (0.5)  loss: {binary_cross_entropy(p_random,  y_true).item():.4f}  (expect ~0.693 = ln2)')
print(f'Completely wrong    loss: {binary_cross_entropy(p_wrong,   y_true).item():.4f}  (expect very large)')

In [ ]:
def binary_accuracy(p_hat, y):
    """
    p_hat : [batch_size, 1] — predicted probabilities
    y     : [batch_size, 1] — true binary labels
    """
    preds = (p_hat >= 0.5).float()   # threshold at 0.5
    return (preds == y).float().mean()

## Training

In [ ]:
net       = BinaryLogisticRegression()
optimizer = optim.SGD(net.parameters(), lr=0.1)
num_epochs = 1000

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

# Labels must be float and shape [N, 1] for BCE
tr_targets  = torch.from_numpy(y_tr.astype('float32')).unsqueeze(1)   # [100, 1]
val_targets = torch.from_numpy(y_val.astype('float32')).unsqueeze(1)  # [100, 1]

def pred_binary(X_np):
    """Returns a 1-D array of P(class=1) for the decision boundary plotter."""
    with torch.no_grad():
        return net(torch.from_numpy(X_np)).numpy()[:, 0]

plot_decision_boundary(pred_binary, X_te, y_te, title='Untrained Binary Logistic Regression')
plt.show()

for e in range(num_epochs):
    # ── Forward ──────────────────────────────────────────
    p_hat_tr = net(torch.from_numpy(X_tr))          # [100, 1]
    tr_loss  = binary_cross_entropy(p_hat_tr, tr_targets)

    # ── Backward ─────────────────────────────────────────
    optimizer.zero_grad()
    tr_loss.backward()
    optimizer.step()

    # ── Validation (no gradients needed) ─────────────────
    with torch.no_grad():
        p_hat_val = net(torch.from_numpy(X_val))
        val_loss  = binary_cross_entropy(p_hat_val, val_targets)
        val_acc   = binary_accuracy(p_hat_val, val_targets)
        train_acc = binary_accuracy(p_hat_tr,  tr_targets)

    train_losses.append(tr_loss.item())
    val_losses.append(val_loss.item())
    train_accs.append(train_acc.item())
    val_accs.append(val_acc.item())

    if e % 100 == 0:
        print(f'Epoch {e:4d}  Train Loss: {tr_loss.item():.3f}  '
              f'Val Loss: {val_loss.item():.3f}  Val Acc: {val_acc.item():.3f}')

# ── Test ──────────────────────────────────────────────────
with torch.no_grad():
    te_targets = torch.from_numpy(y_te.astype('float32')).unsqueeze(1)
    p_hat_te   = net(torch.from_numpy(X_te))
    te_loss    = binary_cross_entropy(p_hat_te, te_targets)
    te_acc     = binary_accuracy(p_hat_te, te_targets)

print(f'\nTest Loss: {te_loss.item():.3f}  Test Accuracy: {te_acc.item():.3f}')

In [ ]:
plot_decision_boundary(pred_binary, X_te, y_te, title='Trained Binary Logistic Regression (Linear Boundary)')
plt.show()

epoch = np.arange(num_epochs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epoch, train_losses, 'r', label='Train')
axes[0].plot(epoch, val_losses,   'b', label='Val')
axes[0].set_title('Binary Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epoch, train_accs, 'r', label='Train')
axes[1].plot(epoch, val_accs,   'b', label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

### What you should observe

The boundary is a **straight line** — the model is doing exactly what it is supposed to do. It cannot follow the curved half-moon shape, so accuracy plateaus well below 100%. This is not a training failure; it is the fundamental limit of a linear classifier.

With only **3 parameters** ($w_1, w_2, b$), the entire model is captured by:

$$\hat{p} = \sigma(w_1 x_1 + w_2 x_2 + b)$$

Let's visualise that line explicitly.

In [ ]:
w1 = net.W.data[0, 0].item()
w2 = net.W.data[0, 1].item()
b  = net.b.data[0].item()
print(f'Learned: w1={w1:.3f}  w2={w2:.3f}  b={b:.3f}')

# Decision boundary: w1*x1 + w2*x2 + b = 0  →  x2 = -(w1*x1 + b) / w2
x1_range = np.linspace(X_te[:, 0].min() - 0.5, X_te[:, 0].max() + 0.5, 100)
x2_boundary = -(w1 * x1_range + b) / w2

plt.figure()
plt.scatter(X_te[:, 0], X_te[:, 1], c=-y_te, cmap=plt.cm.Spectral, s=40)
plt.plot(x1_range, x2_boundary, 'k--', linewidth=2, label=f'$w_1 x_1 + w_2 x_2 + b = 0$')
plt.legend()
plt.title('Explicit linear decision boundary')
plt.show()

---
# Part 2 — From Binary to Multi-class: The Softmax Formulation

## Why change anything?

Binary logistic regression works well for two classes, but it does not generalise to three or more. The standard extension is to produce **one output score per class** and use **softmax** to convert those scores into a probability distribution.

## Comparing the two formulations

For exactly **two** classes the two are mathematically equivalent:

| | Binary LR | Softmax LR (2-class) |
|---|---|---|
| Outputs | 1 scalar $\hat{p}$ | 2 values $[\hat{p}_0, \hat{p}_1]$, sum to 1 |
| Activation | sigmoid | softmax |
| Loss | Binary Cross-Entropy | Categorical Cross-Entropy |
| Label format | scalar $y \in \{0, 1\}$ | one-hot vector |
| Parameters | $w \in \mathbb{R}^d$, $b \in \mathbb{R}$ | $W \in \mathbb{R}^{2 \times d}$, $b \in \mathbb{R}^2$ |

The softmax version has **twice as many parameters** but they are redundant: $\hat{p}_0 = 1 - \hat{p}_1$ always. Both models learn the same line. Softmax is preferred in practice because it scales to any number of classes without changing the architecture.

In [ ]:
num_output = 2

class SoftmaxLogisticRegression(nn.Module):
    """
    Two-class logistic regression using softmax.
    Produces one probability per class; they always sum to 1.
    """

    def __init__(self):
        super(SoftmaxLogisticRegression, self).__init__()
        # TWO output units — one score per class
        # W shape: [num_output, num_features] = [2, 2]
        self.W = nn.Parameter(torch.randn(num_output, num_features))
        self.b = nn.Parameter(torch.randn(num_output))

    def forward(self, x):
        z = F.linear(x, self.W, self.b)   # [batch, 2]
        return F.softmax(z, dim=1)         # [batch, 2] — rows sum to 1


net2 = SoftmaxLogisticRegression()
print('W shape:', net2.W.shape)   # [2, 2]
print('b shape:', net2.b.shape)   # [2]

out = net2(torch.randn(4, num_features))
print('\nOutput shape:', out.shape)                      # [4, 2]
print('Row sums (must be 1.0):', out.sum(dim=1).data)   # all 1.0

## Loss: Categorical Cross-Entropy

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \sum_{c=0}^{C-1} t_{ic} \log(\hat{p}_{ic})$$

With two classes and one-hot targets this reduces to exactly the same expression as BCE. The two losses are equivalent here — the only difference is the label format.

In [ ]:
def cross_entropy(ys, ts):
    """
    ys : [batch, num_classes] — softmax probabilities
    ts : [batch, num_classes] — one-hot targets
    """
    return -torch.mean(torch.sum(ts * torch.log(ys + 1e-7), dim=1))

def softmax_accuracy(ys, ts):
    preds   = torch.max(ys, 1)[1]
    targets = torch.max(ts, 1)[1]
    return (preds == targets).float().mean()

In [ ]:
net2       = SoftmaxLogisticRegression()
optimizer2 = optim.SGD(net2.parameters(), lr=0.1)

# Labels are now one-hot: class 0 → [1, 0], class 1 → [0, 1]
tr_targets2  = torch.from_numpy(onehot(y_tr,  num_output)).float()
val_targets2 = torch.from_numpy(onehot(y_val, num_output)).float()

train_losses2, val_losses2 = [], []
train_accs2,   val_accs2   = [], []

def pred_softmax(X_np):
    with torch.no_grad():
        return net2(torch.from_numpy(X_np)).numpy()[:, 1]   # P(class=1)

for e in range(num_epochs):
    ys_tr   = net2(torch.from_numpy(X_tr))
    tr_loss = cross_entropy(ys_tr, tr_targets2)

    optimizer2.zero_grad()
    tr_loss.backward()
    optimizer2.step()

    with torch.no_grad():
        ys_val    = net2(torch.from_numpy(X_val))
        val_loss  = cross_entropy(ys_val, val_targets2)
        val_acc   = softmax_accuracy(ys_val, val_targets2)
        train_acc = softmax_accuracy(ys_tr,  tr_targets2)

    train_losses2.append(tr_loss.item())
    val_losses2.append(val_loss.item())
    train_accs2.append(train_acc.item())
    val_accs2.append(val_acc.item())

    if e % 100 == 0:
        print(f'Epoch {e:4d}  Train Loss: {tr_loss.item():.3f}  '
              f'Val Loss: {val_loss.item():.3f}  Val Acc: {val_acc.item():.3f}')

with torch.no_grad():
    ys_te       = net2(torch.from_numpy(X_te))
    te_targets2 = torch.from_numpy(onehot(y_te, num_output)).float()
    te_loss2    = cross_entropy(ys_te, te_targets2)
    te_acc2     = softmax_accuracy(ys_te, te_targets2)

print(f'\nTest Loss: {te_loss2.item():.3f}  Test Accuracy: {te_acc2.item():.3f}')

In [ ]:
# Side-by-side comparison of decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pred_fn, title in [
    (axes[0], pred_binary,  'Part 1: Binary LR\n(sigmoid, 1 output, BCE)'),
    (axes[1], pred_softmax, 'Part 2: Softmax LR\n(softmax, 2 outputs, CE)'),
]:
    x_min, x_max = X_te[:, 0].min() - .5, X_te[:, 0].max() + .5
    y_min, y_max = X_te[:, 1].min() - .5, X_te[:, 1].max() + .5
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.01).astype('float32'),
        np.arange(y_min, y_max, 0.01).astype('float32'),
    )
    Z = pred_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=plt.cm.RdBu)
    ax.scatter(X_te[:, 0], X_te[:, 1], c=-y_te, cmap=plt.cm.Spectral, s=20)
    ax.set_title(title)

plt.suptitle('Both models learn the same linear boundary — different formulation, same result', fontsize=12)
plt.tight_layout()
plt.show()

The two boundaries should look nearly identical — confirming that for binary problems the formulations are equivalent. From here, the softmax version scales directly to 3, 10, or 1000 classes just by changing `num_output`.

---
# Assignments

### Assignment 1 — Understand the sigmoid
Plot $\sigma(z)$ for $z \in [-6, 6]$. Mark the point where $\sigma(z) = 0.5$. Then print the raw pre-activation values $z = Wx + b$ alongside $\hat{p} = \sigma(z)$ for 10 test samples. For which values of $z$ is the model most confident? Least confident?

---

### Assignment 2 — Decision boundary geometry
The decision boundary is exactly the line $w_1 x_1 + w_2 x_2 + b = 0$. Extract the learned parameters from `net` and draw this line manually on a scatter plot of the test data (see the explicit boundary cell above for a starter). Then do the same for `net2` using its weight difference $W[1] - W[0]$. Do they produce the same line?

---

### Assignment 3 — BCE vs CE equivalence
Train both models with the same random seed (`torch.manual_seed(0)`) and learning rate. After 1000 epochs compare their final validation accuracies and the slopes of their decision boundaries. Are they the same? If not, what explains the difference even though the formulations are mathematically equivalent?

---

### Assignment 4 — Effect of learning rate
Re-run the binary model (`BinaryLogisticRegression`) with learning rates `[0.001, 0.01, 0.1, 1.0]`. Plot all four validation loss curves on the same figure. Which converges fastest? Which diverges or oscillates? What does this tell you about the relationship between step size and convergence?

---

### Assignment 5 — The accuracy ceiling
No matter how you tune the learning rate, epochs, or optimizer, logistic regression on this dataset will not exceed ~87% accuracy. Write 3–5 sentences explaining *why* this ceiling exists. What property of the data causes it? What property of the model is the limiting factor? What would you need to change to break through it?

In [ ]:
# Your code here
